# Model Registry: Train, Register, and Serve R Models

This notebook demonstrates the `snowflakeR` package workflow for registering R models
in the Snowflake Model Registry and running inference.

**What you'll do:**
1. Train models in R (as usual)
2. Test predictions locally
3. Register to Snowflake Model Registry with one function call
4. Manage versions and metrics
5. Track experiments with tidymodels integration
6. Run remote inference via SPCS

**Under the hood:** `snowflakeR` auto-generates a Python `CustomModel` wrapper
that uses `rpy2` to load and call your R model. You never write Python.

This notebook is for **Snowflake Workspace Notebooks** (Python kernel + `%%R` magic).
For local environments (RStudio, Posit, JupyterLab), use `local_model_registry.ipynb`.

**Before you start:** Optionally edit `snowflaker_config.yaml` to set your
database, schema, warehouse, and EAI settings.

**Sections:**
1. Setup
2. Connect
3. Version Compatibility
4. Train a Model
5. Test Locally
6. Register to Snowflake
7. Manage Models
8. Experiment Tracking
9. Inference
10. Advanced: Custom Predict Code
11. Cleanup


In [ ]:
import time as _time
_nb_start = _time.time()
print(f"Notebook started: {_time.strftime('%Y-%m-%d %H:%M:%S')}")

## 1. Setup

Single cell to set session context, validate EAI, install R, and install snowflakeR.

In [ ]:
from sfnb_setup import setup_notebook
setup_notebook(config="snowflaker_config.yaml", packages=["snowflakeR"])

## 2. Connect

Session context was set by `setup_notebook()` above.
All table references use fully qualified names via `sfr_fqn()`.

In [ ]:
%%R
library(snowflakeR)

# Connect (auto-detects Workspace session; context set by setup_notebook)
conn <- sfr_connect()
conn

# Create a Model Registry context
reg <- sfr_model_registry(conn)
reg

---

## 3. Version Compatibility

Different `snowflake-ml-python` versions enable different features.
`snowflakeR` checks the installed version at runtime and provides clear
error messages when a feature requires a newer SDK.


In [ ]:
%%R
# Check the installed snowflake-ml-python version
ml_ver <- sfr_ml_version()
rcat(sprintf("snowflake-ml-python version: %s", ml_ver))

# See which snowflakeR features are available with this version
sfr_check_features(conn)


---
## 4. Train a Model

### Example A: Linear regression on mtcars

Train any R model as you normally would -- nothing snowflakeR-specific here.

In [ ]:
%%R
# Train a simple linear model
model_lm <- lm(mpg ~ wt + hp + cyl, data = mtcars)
summary(model_lm)

### Example B: Time series with forecast

```r
%%R
library(forecast)
model_arima <- auto.arima(AirPassengers)
summary(model_arima)
```

---

## 5. Test Locally

`sfr_predict_local()` runs the **exact same prediction logic** that will execute
inside Snowflake, but entirely in R. Use this to verify before registering.

In [ ]:
%%R
# Create test data
test_data <- data.frame(
  wt  = c(2.5, 3.0, 3.5, 4.0),
  hp  = c(110, 150, 200, 245),
  cyl = c(4L, 6L, 8L, 8L)
)

# Test locally -- same predict path as remote
preds <- sfr_predict_local(model_lm, test_data)
cbind(test_data, preds)

---
## 6. Register to Snowflake

One call to `sfr_log_model()` handles everything:
- Saves the R model to `.rds`
- Auto-generates a Python `CustomModel` wrapper
- Registers in the Snowflake Model Registry

In [ ]:
%%R
mv <- sfr_log_model(
  reg,
  model        = model_lm,
  model_name   = "SFR_DEMO_MPG",
  version_name = "V1",
  input_cols   = list(wt = "double", hp = "double", cyl = "integer"),
  output_cols  = list(prediction = "double"),
  comment      = "Linear regression: MPG from weight, horsepower, cylinders",
  task         = "TABULAR_REGRESSION"
)

mv

### Key parameters for `sfr_log_model()`

| Parameter | Description |
|-----------|-------------|
| `model` | Any R object that can be `saveRDS()`'d |
| `model_name` | Registry name (uppercase recommended) |
| `input_cols` | Named list: column name -> type (`double`, `integer`, `string`, `boolean`) |
| `output_cols` | Named list: output column name -> type |
| `predict_fn` | R function name (default: `"predict"`) |
| `predict_pkgs` | R packages needed at inference time |
| `conda_deps` | Extra conda packages (`r-base`, `rpy2`, and `numpy<2.0` always included) |
| `pin_versions` | Auto-pin R + package versions from current session (default `TRUE`) |
| `target_platforms` | `"SNOWPARK_CONTAINER_SERVICES"` (default) or `"WAREHOUSE"` |

> **Auto version pinning:** By default, `sfr_log_model()` snapshots the current
> R version and all `predict_pkgs` (including tidymodels sub-packages) and pins
> them as exact versions in `conda_deps`.  This ensures the SPCS container
> environment matches your training session.  Set `pin_versions = FALSE` to
> use floating constraints.
>
> **Container note:** The default `conda_deps` include `numpy<2.0`.
> This is required because without it the SPCS conda solver resolves to
> Python 3.12 + numpy 2.x for R model containers, which triggers a
> server-side `recarray has no attribute fillna` crash. Pinning `numpy<2.0`
> causes the solver to select Python 3.11 (matching pure Python model
> containers), which avoids the bug entirely.

---

## 7. Manage Models

### List and inspect models

In [ ]:
%%R
# List all models in the registry
models <- sfr_show_models(reg)
models

In [ ]:
%%R
# Get a specific model
m <- sfr_get_model(reg, "SFR_DEMO_MPG")
m

# Show versions
sfr_show_model_versions(reg, "SFR_DEMO_MPG")

### Metrics

Attach evaluation metrics to model versions:

In [ ]:
%%R
# Calculate and set metrics
preds_train <- predict(model_lm, mtcars)
rmse <- sqrt(mean((mtcars$mpg - preds_train)^2))
r_sq <- summary(model_lm)$r.squared

sfr_set_model_metric(reg, "SFR_DEMO_MPG", "V1", "rmse", rmse)
sfr_set_model_metric(reg, "SFR_DEMO_MPG", "V1", "r_squared", r_sq)

rcat(sprintf("RMSE: %.3f, R-squared: %.3f", rmse, r_sq))

In [ ]:
%%R
# Retrieve metrics
sfr_show_model_metrics(reg, "SFR_DEMO_MPG", "V1")

### Log a second version

In [ ]:
%%R
# Train a better model (added displacement)
model_v2 <- lm(mpg ~ wt + hp + cyl + disp, data = mtcars)

mv2 <- sfr_log_model(
  reg,
  model        = model_v2,
  model_name   = "SFR_DEMO_MPG",
  version_name = "V2",
  input_cols   = list(wt = "double", hp = "double", cyl = "integer", disp = "double"),
  output_cols  = list(prediction = "double"),
  comment      = "V2: added displacement"
)

# Set as default
sfr_set_default_model_version(reg, "SFR_DEMO_MPG", "V2")

---

## 8. Experiment Tracking

Snowflake Experiment Tracking lets you log hyperparameter runs and compare
models directly in Snowsight. This is complementary to registry metrics:

- **Experiments** = training-time comparison (which hyperparams were best?)
- **Registry metrics** = production monitoring (how does the deployed model perform?)

`snowflakeR` provides a full R API plus `tidymodels` integration that automatically
logs `tune_grid()` / `tune_bayes()` results as Snowflake experiment runs.


### Manual experiment workflow

The basic lifecycle: create an experiment, start runs, log parameters and metrics,
end runs.


In [ ]:
%%R
# Create an experiment
exp <- sfr_experiment(conn, name = "SFR_DEMO_MPG_EXPERIMENT")
exp


In [ ]:
%%R
# Run 1: simple model (wt + hp)
sfr_start_run(exp, name = "simple_model")

sfr_exp_log_params(exp, formula = "mpg ~ wt + hp", n_predictors = 2)

m1 <- lm(mpg ~ wt + hp, data = mtcars)
rmse1 <- sqrt(mean(residuals(m1)^2))
r2_1  <- summary(m1)$r.squared

sfr_exp_log_metrics(exp, rmse = rmse1, r_squared = r2_1)
sfr_end_run(exp)

rcat(sprintf("Run 1 -- RMSE: %.3f, R²: %.3f", rmse1, r2_1))


In [ ]:
%%R
# Run 2: fuller model (wt + hp + cyl + disp)
sfr_start_run(exp, name = "full_model")

sfr_exp_log_params(exp, formula = "mpg ~ wt + hp + cyl + disp", n_predictors = 4)

m2 <- lm(mpg ~ wt + hp + cyl + disp, data = mtcars)
rmse2 <- sqrt(mean(residuals(m2)^2))
r2_2  <- summary(m2)$r.squared

sfr_exp_log_metrics(exp, rmse = rmse2, r_squared = r2_2)
sfr_end_run(exp)

rcat(sprintf("Run 2 -- RMSE: %.3f, R²: %.3f", rmse2, r2_2))


### tidymodels integration

`sfr_experiment_from_tune()` takes a `tune_results` object and logs every
hyperparameter configuration as a separate Snowflake experiment run --
equivalent to Python's autologging, but for the R ecosystem.

`sfr_experiment_log_best()` selects the best configuration, fits the final
model, and registers it in the Model Registry.


In [ ]:
%%R
# tidymodels experiment tracking
# (requires: tidymodels, ranger -- skip gracefully if not installed)
if (requireNamespace("tidymodels", quietly = TRUE) &&
    requireNamespace("ranger", quietly = TRUE)) {

  library(tidymodels)

  # Define a ranger random forest workflow on mtcars
  rf_spec <- rand_forest(
    mtry  = tune(),
    trees = tune()
  ) |>
    set_engine("ranger") |>
    set_mode("regression")

  rf_wf <- workflow() |>
    add_formula(mpg ~ wt + hp + cyl + disp) |>
    add_model(rf_spec)

  # Small grid search with 3-fold CV
  set.seed(42)
  rf_grid <- grid_regular(mtry(range = c(2, 4)), trees(range = c(50, 200)), levels = 3)
  folds   <- vfold_cv(mtcars, v = 3)

  tune_results <- tune_grid(rf_wf, resamples = folds, grid = rf_grid)

  # Log ALL tuning runs to the Snowflake experiment
  sfr_experiment_from_tune(exp, tune_results, prefix = "rf")
  rcat("All tune_grid configurations logged to Snowflake experiment.")

  # Log the best model and register it
  sfr_experiment_log_best(
    exp, tune_results, rf_wf,
    model_name = "SFR_DEMO_MPG_RF",
    metric     = "rmse",
    train_data = mtcars,
    input_cols = list(wt = "double", hp = "double", cyl = "integer", disp = "double"),
    output_cols = list(prediction = "double")
  )
  rcat("Best model registered in Model Registry.")

} else {
  rcat("Skipping tidymodels demo (tidymodels or ranger not installed).")
  rcat("Install with: install.packages(c('tidymodels', 'ranger'))")
}


In [ ]:
%%R
# Clean up experiment
sfr_delete_experiment(exp)
rcat("Experiment deleted.")


---
## 9. Inference

### Local prediction (works everywhere)

`sfr_predict_local()` runs the **exact same prediction logic** that the registered model
uses, but entirely in your local R session. Use this to verify predictions before
deploying to production.

> **Note:** R models require `rpy2` and `r-base` at inference time, which are
> not available in the Snowflake warehouse Anaconda channel. Therefore, warehouse
> inference (`sfr_predict()` without a service) is not supported for R models.
> For production inference in Snowflake, deploy via SPCS (see below).

In [ ]:
%%R
# Test data
new_data <- data.frame(
  wt   = c(2.62, 3.44, 3.57),
  hp   = c(110, 175, 245),
  cyl  = c(4L, 6L, 8L),
  disp = c(120.3, 258.0, 360.0)
)

# Predict locally -- same logic as the registered model
preds <- sfr_predict_local(model_lm, new_data)
cbind(new_data, preds)

### Production inference via SPCS

For production inference **inside Snowflake**, deploy the model as a
Snowpark Container Services (SPCS) service. This creates a container with
R, rpy2, and your model, then serves predictions via a REST endpoint.

The cells below create the SPCS infrastructure (compute pool + image repo)
if it doesn't already exist, then deploy, wait, test, benchmark, and undeploy.
Each step is a separate cell so you can re-run individually.

In [ ]:
%%R
# -- 0. Create SPCS infrastructure (if not exists) --
sfr_create_compute_pool(conn, "R_FORECAST_POOL", instance_family = "CPU_X64_M")
sfr_create_image_repo(conn, sfr_fqn(conn, "R_FORECAST_IMAGES"))

# -- 1. Deploy as SPCS service (force = TRUE drops existing service first)
sfr_deploy_model(
  reg,
  model_name   = "SFR_DEMO_MPG",
  version_name = "V2",
  service_name = "mpg_service",
  compute_pool = "R_FORECAST_POOL",
  image_repo   = sfr_fqn(conn, "R_FORECAST_IMAGES"),
  force        = TRUE
)

In [ ]:
%%R
# -- 2. Check service status (one-off) --
st <- sfr_get_service_status(reg, "mpg_service")
rcat(sprintf("Status: %s | Message: %s | FQN: %s", st$status, st$message, st$fqn))
if (!is.null(st$containers)) rprint(st$containers[, c("containerName", "status", "message")])

In [ ]:
%%R
# -- 3. Wait for service to be ready --
# Polls every 15 seconds, times out after 10 minutes
sfr_wait_for_service(reg, "mpg_service", timeout_min = 10, poll_sec = 15)

In [ ]:
%%R
# -- 4. Run inference via the service (Python bridge path) --
preds <- sfr_predict(reg, "SFR_DEMO_MPG", new_data, service_name = "mpg_service")
preds

### Direct REST Inference (faster, no Python bridge)

`sfr_predict_rest()` calls the SPCS model endpoint directly from R using `httr2`,
bypassing the Python/rpy2 bridge entirely for the lowest latency.

**Auth:** In Workspace Notebooks the SPCS OAuth token is used automatically.
Outside Workspace, set `SNOWFLAKE_PAT` or pass a connection with key-pair auth
for auto-JWT generation.

In [ ]:
%%R
# -- 5a. Discover the REST endpoint --
ep <- sfr_service_endpoint(conn, "mpg_service")
ep

In [ ]:
%%R
# -- 5b. REST predict (pure R, no Python bridge) --
# Auth is automatic in Workspace (SPCS OAuth token).
# Outside Workspace: uses PAT or JWT from key-pair auth.
rest_preds <- sfr_predict_rest(ep, new_data, conn = conn)
rest_preds

In [ ]:
%%R
# -- 5c. Benchmark REST inference --
rest_bench <- sfr_benchmark_rest(ep, new_data, n = 20, conn = conn)
cat("\nREST  median:", round(rest_bench$median_sec, 3), "s\n")

In [ ]:
%%R
# -- 6. Undeploy when done --
sfr_undeploy_model(reg, "SFR_DEMO_MPG", "V2", "mpg_service")

## 10. Advanced -- Custom Predict Code

For models that need special prediction logic (e.g., `forecast`, multi-step pipelines),
use the `predict_body` template.

### Template variables

| Variable | Description |
|----------|-------------|
| `{{MODEL}}` | The loaded R model object |
| `{{INPUT}}` | The input data.frame |
| `{{UID}}` | Unique ID for variable naming |
| `{{N}}` | Number of rows in input |

In [ ]:
%%R
# Example: forecast model with custom output
# (Uncomment and adapt for your use case)

# library(forecast)
# arima_model <- auto.arima(AirPassengers)
#
# mv_forecast <- sfr_log_model(
#   reg,
#   model        = arima_model,
#   model_name   = "SFR_DEMO_FORECAST",
#   predict_fn   = "forecast",
#   predict_pkgs = c("forecast"),
#   predict_body = '
#     pred_{{UID}} <- forecast({{MODEL}}, h = {{N}})
#     result_{{UID}} <- data.frame(
#       period = seq_len({{N}}),
#       point_forecast = as.numeric(pred_{{UID}}$mean),
#       lower_95 = as.numeric(pred_{{UID}}$lower[,2]),
#       upper_95 = as.numeric(pred_{{UID}}$upper[,2])
#     )
#   ',
#   input_cols  = list(period = "integer"),
#   output_cols = list(
#     period = "integer", point_forecast = "double",
#     lower_95 = "double", upper_95 = "double"
#   )
# )

### Recommended: `sfr_predict_body()` workflow

Instead of writing raw template strings with `{{MODEL}}` placeholders, use
`sfr_predict_body()` to convert a **normal R function** into the template.
This lets you write, test, and debug your predict logic as regular R code
before registering.

The workflow:

1. Write a function with `(model, input)` formals
2. Test it directly as a regular R function
3. Convert to a template with `sfr_predict_body()`
4. Pre-flight the template with `sfr_predict_local()`
5. Register with `sfr_log_model(predict_body = ...)`

In [ ]:
%%R
# Step 1: Write predict logic as a normal R function
my_predict <- function(model, input) {
  raw   <- predict(model, newdata = input, interval = "prediction")
  result <- data.frame(
    prediction = raw[, "fit"],
    lower_95   = raw[, "lwr"],
    upper_95   = raw[, "upr"]
  )
}

# Step 2: Test it directly (plain R -- no templates involved)
test_data <- data.frame(wt = c(2.5, 3.0, 3.5),
                        hp = c(110, 150, 200),
                        cyl = c(4L, 6L, 8L))

my_predict(model_lm, test_data)
rcat("Direct function call works!")

# Step 3: Convert to template
body_str <- sfr_predict_body(my_predict)
rcat("Generated template:")
cat(body_str)

# Step 4: Pre-flight the template via sfr_predict_local
# This executes the EXACT same gsub + eval(parse()) path that
# SPCS will use at inference time.
preds_preflight <- sfr_predict_local(
  model_lm, test_data,
  predict_body = body_str
)

rcat("Pre-flight results (template execution):")
cbind(test_data, preds_preflight)

---

## 11. Cleanup

In [ ]:
%%R
# Uncomment to clean up demo objects
# (commented out to avoid accidental deletion on Run All)
#
# sfr_delete_model(reg, "SFR_DEMO_MPG")
# sfr_delete_model(reg, "SFR_DEMO_MPG_RF")
# sfr_execute(conn, paste("DROP TABLE IF EXISTS", sfr_fqn(conn, "SFR_DEMO_PREDICT_INPUT")))
# sfr_disconnect(conn)
# rcat("Cleanup complete.")

---

## Supported model types

Any R model that can be serialised with `saveRDS()` works:

- `lm()`, `glm()` (base R)
- `randomForest::randomForest()`
- `xgboost::xgb.train()`
- `ranger::ranger()`
- `forecast::auto.arima()`, `forecast::ets()`
- `tidymodels` workflows
- Custom S3/S4 model objects

## Next steps

- **Model Consumption:** See `workspace_model_consumption.ipynb` for SQL-direct
  batch inference, model aliases, and training Python models from R.
- **Model Monitoring:** See `workspace_model_monitoring.ipynb` for drift detection
  and performance tracking.
- **Feature Store:** See `workspace_feature_store.ipynb` for feature engineering
  with time aggregation and online serving.

In [ ]:
_nb_elapsed = _time.time() - _nb_start
_mins, _secs = divmod(_nb_elapsed, 60)
print(f"Notebook completed: {_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total execution time: {int(_mins)}m {_secs:.1f}s")

In [ ]:
# CI results -- writes a results table when run non-interactively
try:
    from snowflake.snowpark.context import get_active_session
    _ci_session = get_active_session()
    _ci_session.sql("""
        CREATE OR REPLACE TABLE NOTEBOOK_CI.MODEL_REGISTRY_RESULTS AS
        SELECT 'model_registry' AS TEST_NAME, 'PASS' AS STATUS,
               CURRENT_TIMESTAMP() AS RUN_AT, CURRENT_USER() AS RUN_BY
    """).collect()
    print("CI results written to NOTEBOOK_CI.MODEL_REGISTRY_RESULTS")
except Exception:
    pass